# Full Post-Quantum Signal Protocol Integration

This notebook combines the three post-quantum cryptographic enhancements into a single, cohesive simulation of the Signal Protocol:
1. **LatticeSTS**: For long-term Identity Key exchange.
2. **LWE/SIS Hybridization**: For ephemeral (One-Time and Signed Pre-Key) key exchanges during the X3DH handshake.
3. **Module-LWE**: For the continuous asymmetric ratchet (Double Ratchet).

In [21]:
import numpy as np
import time
import hashlib
import math
import hmac
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes

In [22]:
class LatticeSTS:
    def __init__(self, n=256, seed=123):
        self.n = n
        self.q = 2 * (n**2) + 1
        self.m = 2 * n
        self.beta = int(np.sqrt(self.m))
        self.rng = np.random.RandomState(seed)
        self.A = self.rng.randint(0, self.q, (self.m, self.m))
        
    def generate_keypair_alice(self, user_seed):
        rng = np.random.RandomState(user_seed)
        s_A = rng.randint(-self.beta//2, self.beta//2, self.m)
        p_A = (self.A @ s_A) % self.q
        return s_A, p_A
        
    def generate_keypair_bob(self, user_seed):
        rng = np.random.RandomState(user_seed)
        s_B = rng.randint(-self.beta//2, self.beta//2, self.m)
        p_B = (s_B @ self.A) % self.q
        return s_B, p_B
        
    def compute_shared_secret_alice(self, s_A, p_B):
        val = (p_B @ s_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

    def compute_shared_secret_bob(self, s_B, p_A):
        val = (s_B @ p_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

class LWESIS:
    def __init__(self, n=64, m=128, q=10007, alpha=0.0001, seed=42):
        self.n = n
        self.m = m
        self.q = q
        self.alpha = alpha
        self.rng = np.random.RandomState(seed)
        self.M = self.rng.randint(0, q, (n, m))
        
    def discrete_gaussian(self, rng, size):
        return np.round(rng.normal(0, self.alpha * self.q, size)).astype(int) % self.q
        
    def signal_function(self, y):
        return (y > (self.q // 4)) and (y < (3 * self.q // 4))
        
    def robust_extractor(self, x, sigma):
        return ((x + sigma * ((self.q - 1) // 2)) % self.q) % 2
        
    def generate_keypair_alice(self, user_seed):
        rng = np.random.RandomState(user_seed)
        s_A = rng.randint(0, self.q, (1, self.n))
        e_A = self.discrete_gaussian(rng, (1, self.m))
        P_A = (s_A @ self.M + 2 * e_A) % self.q
        return s_A, P_A
        
    def generate_keypair_bob(self, user_seed):
        rng = np.random.RandomState(user_seed)
        s_B = self.discrete_gaussian(rng, (self.m, 1))
        P_B = (self.M @ s_B) % self.q
        return s_B, P_B
        
    def compute_shared_secret_alice(self, s_A, P_B):
        K_A = (s_A @ P_B) % self.q
        sigma = self.signal_function(K_A[0,0])
        bit = self.robust_extractor(K_A[0,0], sigma)
        return bit, sigma
        
    def compute_shared_secret_bob(self, s_B, P_A, sigma):
        K_B = (P_A @ s_B) % self.q
        bit = self.robust_extractor(K_B[0,0], sigma)
        return bit

class ModuleLWE:
    def __init__(self, n=256, k=3, q=7681, eta=2, seed=789):
        self.n = n
        self.k = k
        self.q = q
        self.eta = eta
        self.rng = np.random.RandomState(seed)
        self.A = self.rng.randint(0, q, (k, k, n))
        
    def cbd(self, rng, size):
        x = rng.randint(0, 2, (self.eta, *size))
        y = rng.randint(0, 2, (self.eta, *size))
        return np.sum(x - y, axis=0)

    def poly_mul(self, poly1, poly2):
        res = np.convolve(poly1, poly2)
        n = self.n
        final_res = (res[:n] - np.concatenate([res[n:], [0]*(n - (len(res)-n))])) % self.q
        return final_res

    def generate_keypair(self, user_seed):
        rng = np.random.RandomState(user_seed)
        s = self.cbd(rng, (self.k, self.n))
        e = self.cbd(rng, (self.k, self.n))
        t = np.zeros((self.k, self.n), dtype=int)
        for i in range(self.k):
            for j in range(self.k):
                t[i] = (t[i] + self.poly_mul(self.A[i, j], s[j])) % self.q
            t[i] = (t[i] + e[i]) % self.q
        return s, t

    def encapsulate(self, user_seed, t_pub):
        rng = np.random.RandomState(user_seed)
        r = self.cbd(rng, (self.k, self.n))
        e1 = self.cbd(rng, (self.k, self.n))
        e2 = self.cbd(rng, (self.n,))
        u = np.zeros((self.k, self.n), dtype=int)
        for i in range(self.k):
            for j in range(self.k):
                u[i] = (u[i] + self.poly_mul(self.A[j, i], r[j])) % self.q
            u[i] = (u[i] + e1[i]) % self.q
        v = np.zeros((self.n,), dtype=int)
        for j in range(self.k):
            v = (v + self.poly_mul(t_pub[j], r[j])) % self.q
        v = (v + e2) % self.q
        m = rng.randint(0, 2, self.n)
        v = (v + m * (self.q // 2)) % self.q
        return (u, v), m

    def decapsulate(self, s_priv, u, v):
        temp = np.zeros((self.n,), dtype=int)
        for j in range(self.k):
            temp = (temp + self.poly_mul(s_priv[j], u[j])) % self.q
        res = (v - temp) % self.q
        bits = (res > (self.q // 4)) & (res < (3 * self.q // 4))
        return bits.astype(int)


In [23]:
class PQKeyBundle:
    def __init__(self, base_seed):
        self.sts = LatticeSTS()
        self.hybrid = LWESIS()
        self.mlwe = ModuleLWE()
        self.base_seed = base_seed
        
        self.ik_priv, self.ik_pub = None, None
        self.spk_priv, self.spk_pub = None, None
        self.otk_priv, self.otk_pub = None, None
        
    def generate_alice(self):
        self.ik_priv, self.ik_pub = self.sts.generate_keypair_alice(self.base_seed + 1)
        self.spk_priv, self.spk_pub = self.hybrid.generate_keypair_alice(self.base_seed + 2)
        self.otk_priv, self.otk_pub = self.hybrid.generate_keypair_alice(self.base_seed + 3)
        
    def generate_bob(self):
        self.ik_priv, self.ik_pub = self.sts.generate_keypair_bob(self.base_seed + 1)
        self.spk_priv, self.spk_pub = self.hybrid.generate_keypair_bob(self.base_seed + 2)
        self.otk_priv, self.otk_pub = self.hybrid.generate_keypair_bob(self.base_seed + 3)
        
    def get_public_bundle(self):
        return {
            'ik': self.ik_pub,
            'spk': self.spk_pub,
            'otk': self.otk_pub
        }


In [24]:
# 1. Setup Phase
alice = PQKeyBundle(base_seed=1000)
alice.generate_alice()

bob = PQKeyBundle(base_seed=2000)
bob.generate_bob()

bob_bundle = bob.get_public_bundle()
alice_bundle = alice.get_public_bundle()

# 2. Post-Quantum X3DH Handshake
start_time = time.time()

# Alice computes
alice_dh1 = alice.sts.compute_shared_secret_alice(alice.ik_priv, bob_bundle['ik'])
alice_dh3_bit, sigma3 = alice.hybrid.compute_shared_secret_alice(alice.spk_priv, bob_bundle['spk'])
alice_dh4_bit, sigma4 = alice.hybrid.compute_shared_secret_alice(alice.otk_priv, bob_bundle['otk'])

alice_master_secret = alice_dh1 + bytes([alice_dh3_bit, alice_dh4_bit])
print(f'Alice Master Secret: {alice_master_secret.hex()[:32]}...')

# Bob computes
bob_dh1 = bob.sts.compute_shared_secret_bob(bob.ik_priv, alice_bundle['ik'])
bob_dh3_bit = bob.hybrid.compute_shared_secret_bob(bob.spk_priv, alice_bundle['spk'], sigma3)
bob_dh4_bit = bob.hybrid.compute_shared_secret_bob(bob.otk_priv, alice_bundle['otk'], sigma4)

bob_master_secret = bob_dh1 + bytes([bob_dh3_bit, bob_dh4_bit])
print(f'Bob Master Secret:   {bob_master_secret.hex()[:32]}...')

assert alice_master_secret == bob_master_secret, 'Handshake failed! Master secrets do not match.'
print(f'PQ X3DH Time: {(time.time() - start_time)*1000:.2f} ms\n')

# 3. Key Derivation
def kdf_root(master_secret):
    return HKDF(algorithm=hashes.SHA256(), length=64, salt=b'\x00'*32, info=b'PQ_X3DH').derive(master_secret)

alice_keys = kdf_root(alice_master_secret)
alice_rk, alice_ck = alice_keys[:32], alice_keys[32:]

bob_keys = kdf_root(bob_master_secret)
bob_rk, bob_ck = bob_keys[:32], bob_keys[32:]

# 4. Asymmetric Ratchet with Module-LWE
print('Performing Asymmetric Ratchet Step...')
start_time = time.time()

# Bob generates a new ratchet keypair
s_bob_ratchet, t_bob_ratchet = bob.mlwe.generate_keypair(user_seed=555)

# Alice encapsulates a secret using Bob\'s new key
ct, alice_bits = alice.mlwe.encapsulate(user_seed=666, t_pub=t_bob_ratchet)
u, v = ct
alice_ratchet_secret = hashlib.sha256(alice_bits.tobytes()).digest()

# Bob decapsulates
bob_bits = bob.mlwe.decapsulate(s_priv=s_bob_ratchet, u=u, v=v)
bob_ratchet_secret = hashlib.sha256(bob_bits.tobytes()).digest()

print(f'Alice Ratchet Secret: {alice_ratchet_secret.hex()[:32]}...')
print(f'Bob Ratchet Secret:   {bob_ratchet_secret.hex()[:32]}...')

assert alice_ratchet_secret == bob_ratchet_secret, 'Ratchet failed! Shared secrets do not match.'
print(f'PQ Asymmetric Ratchet Time: {(time.time() - start_time)*1000:.2f} ms')


Alice Master Secret: ee333c61395c7a24a6fba0fd5f2d837b...
Bob Master Secret:   ee333c61395c7a24a6fba0fd5f2d837b...
PQ X3DH Time: 1.49 ms

Performing Asymmetric Ratchet Step...
Alice Ratchet Secret: 4094998c4d9bf11f72fcd42ef30ff358...
Bob Ratchet Secret:   4094998c4d9bf11f72fcd42ef30ff358...
PQ Asymmetric Ratchet Time: 6.64 ms


## Performance and Metric Analysis

In [25]:
def run_benchmarks(iterations=100):
    # 1. Key Generation Metrics
    start = time.time()
    for i in range(iterations):
        tmp_alice = PQKeyBundle(base_seed=i)
        tmp_alice.generate_alice()
    gen_time = (time.time() - start) / iterations

    # 2. Handshake Metrics
    alice_bench = PQKeyBundle(base_seed=1000)
    alice_bench.generate_alice()
    bob_bench = PQKeyBundle(base_seed=2000)
    bob_bench.generate_bob()
    bob_bundle = bob_bench.get_public_bundle()
    alice_bundle = alice_bench.get_public_bundle()

    start = time.time()
    for i in range(iterations):
        # Alice Handshake compute
        a_dh1 = alice_bench.sts.compute_shared_secret_alice(alice_bench.ik_priv, bob_bundle["ik"])
        a_dh3, s3 = alice_bench.hybrid.compute_shared_secret_alice(alice_bench.spk_priv, bob_bundle["spk"])
        a_dh4, s4 = alice_bench.hybrid.compute_shared_secret_alice(alice_bench.otk_priv, bob_bundle["otk"])
        _ = a_dh1 + bytes([a_dh3, a_dh4])
    handshake_time = (time.time() - start) / iterations

    # 3. Ratchet Metrics
    start = time.time()
    for i in range(iterations):
        s_bob_r, t_bob_r = bob_bench.mlwe.generate_keypair(user_seed=i)
        ct, a_bits = alice_bench.mlwe.encapsulate(user_seed=i+1, t_pub=t_bob_r)
        b_bits = bob_bench.mlwe.decapsulate(s_priv=s_bob_r, u=ct[0], v=ct[1])
    ratchet_time = (time.time() - start) / iterations

    # 4. Size Metrics
    ik_size = alice_bench.ik_pub.nbytes
    spk_size = alice_bench.spk_pub.nbytes
    ratchet_pk_size = t_bob_r.nbytes
    ratchet_ct_size = ct[0].nbytes + ct[1].nbytes

    print("=== Performance Metrics (Average over {} iterations) ===".format(iterations))
    print("Key Bundle Generation:    {:.3f} ms".format(gen_time * 1000))
    print("X3DH Handshake Compute:   {:.3f} ms".format(handshake_time * 1000))
    print("Asymmetric Ratchet Step:  {:.3f} ms".format(ratchet_time * 1000))

    print("\n=== Communication Metrics (Size in Bytes) ===")
    print("Identity Public Key:      {} bytes".format(ik_size))
    print("Ephemeral Public Key:     {} bytes".format(spk_size))
    print("Ratchet Public Key (MLWE): {} bytes".format(ratchet_pk_size))
    print("Ratchet Ciphertext (MLWE): {} bytes".format(ratchet_ct_size))
    print("Total Initial Bundle:     {} bytes".format(ik_size + spk_size*2))

run_benchmarks()

=== Performance Metrics (Average over 100 iterations) ===
Key Bundle Generation:    12.328 ms
X3DH Handshake Compute:   0.021 ms
Asymmetric Ratchet Step:  3.112 ms

=== Communication Metrics (Size in Bytes) ===
Identity Public Key:      4096 bytes
Ephemeral Public Key:     1024 bytes
Ratchet Public Key (MLWE): 6144 bytes
Ratchet Ciphertext (MLWE): 8192 bytes
Total Initial Bundle:     6144 bytes
